# E8 - Early-warning benchmark + multi-seed robustness

**Two experiments in one runner** (they share the same rich per-epoch logs).

**Exp 1 - Early-warning benchmark.** The paper shows the feature norm crosses fn*
~62 epochs before NC onset. Here we log FIVE candidate predictors per epoch and
ask, head-to-head, *which one best anticipates NC onset*:
feature norm, training loss, training error (1-acc), weight norm, gradient norm.
For each signal we use **leave-one-seed-out** calibration (threshold = the
signal's value at NC onset, averaged over the other seeds) and measure the lead
(T_NC - T_cross) on the held-out seed, plus the fraction of runs in which the
signal crosses *before* onset (ordering accuracy).

**Exp 2 - Multi-seed robustness.** Set `SEEDS = range(10)` to report fn* mean,
std and 95% CI over 10 seeds for the anchor cell(s).

**Honest-interpretation notes.**
- A *large* lead is not automatically *good*: training error hits ~0 at the
  terminal phase, long before collapse, so it can show a huge but non-specific
  lead. A useful early-warning signal has a positive, **consistent** (low-variance)
  lead that is specific to imminent collapse. Report all signals and let the
  numbers speak; do not pre-select the winner.
- Metrics, models, and the two-phase protocol are identical to the main paper.
- Report whatever the runs produce, including ties or a signal that beats feature
  norm. **Do not fabricate or hand-tune any value.**

> GPU runtime. MLP-5/MNIST is the default anchor (cheap). ResNet-20/MNIST is
> optional (`RUN_RESNET = True`) and much slower. Kaggle: GPU T4 + Internet ON,
> "Save & Run All" for crash safety.

**Runtime preset.** The run cell defaults to a `FAST` preset (Phase-2 = 300, 5 seeds) that finishes one Kaggle commit in ~8-10 h on a T4 while still reaching collapse (MLP-5/MNIST tNC ~340). Set `FAST = False` for the strict full-length protocol (Phase-2 = 600). Disclose the FAST Phase-2 budget if you report those runs.

In [1]:
import torch, torchvision, time, os, sys, math
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
import torchvision.transforms as T

torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if os.path.isdir('/kaggle/working'):
    PLATFORM, SAVE_DIR, DATA_DIR = 'kaggle', '/kaggle/working/', '/kaggle/working/data/'
elif 'google.colab' in sys.modules or os.path.isdir('/content'):
    PLATFORM = 'colab'
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        SAVE_DIR = '/content/drive/MyDrive/nc_outputs/'
        print('Drive mounted ->', SAVE_DIR)
    except Exception as exc:
        print('WARNING: Drive mount failed (%s). Using /content/.' % exc)
        SAVE_DIR = '/content/'
    DATA_DIR = '/content/data/'
else:
    PLATFORM, SAVE_DIR, DATA_DIR = 'local', './', './data/'
os.makedirs(SAVE_DIR, exist_ok=True); os.makedirs(DATA_DIR, exist_ok=True)
assert torch.cuda.is_available(), 'No GPU - enable a GPU runtime.'
print('Platform:', PLATFORM, '| SAVE_DIR:', SAVE_DIR)
print('GPU:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__)

Platform: kaggle | SAVE_DIR: /kaggle/working/
GPU: Tesla T4 | Torch: 2.10.0+cu128


In [2]:
mlp_tf = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
resnet_tf = T.Compose([T.Pad(2), T.ToTensor(), T.Normalize((0.1307,), (0.3081,)),
                       T.Lambda(lambda x: x.repeat(3, 1, 1))])

def make_loaders(transform, batch_train=512, batch_test=1024):
    tr = torchvision.datasets.MNIST(DATA_DIR, train=True,  download=True, transform=transform)
    te = torchvision.datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)
    return (DataLoader(tr, batch_train, shuffle=True,  num_workers=2, pin_memory=True),
            DataLoader(te, batch_test,  shuffle=False, num_workers=2, pin_memory=True))
print('loaders ready')

loaders ready


In [3]:
# ---- Models (identical to the main paper / E6) ----
class MLP(nn.Module):
    def __init__(self, depth=5, width=512, act_cls=nn.ReLU, num_classes=10, in_dim=784):
        super().__init__()
        layers = [nn.Flatten(), nn.Linear(in_dim, width), act_cls()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), act_cls()]
        self.body = nn.Sequential(*layers)
        self.head = nn.Linear(width, num_classes)
        self._feats = None
        self.body.register_forward_hook(lambda m, i, o: setattr(self, '_feats', o.detach()))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu'); nn.init.zeros_(m.bias)
    def forward(self, x): return self.head(self.body(x))
    def get_features(self, x): self(x); return self._feats
    def get_classifier_weights(self): return self.head.weight.detach()

class BasicBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_c)
        self.skip  = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.skip = nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
                                      nn.BatchNorm2d(out_c))
    def forward(self, x):
        return F.relu(self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x))))) + self.skip(x))

class ResNet20(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1  = nn.Conv2d(3, 16, 3, padding=1, bias=False)
        self.bn1    = nn.BatchNorm2d(16)
        self.layer1 = self._make(16, 16, 3, 1)
        self.layer2 = self._make(16, 32, 3, 2)
        self.layer3 = self._make(32, 64, 3, 2)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.fc     = nn.Linear(64, num_classes)
        self._feats = None
        self.pool.register_forward_hook(lambda m, i, o: setattr(self, '_feats', o.flatten(1).detach()))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):   nn.init.kaiming_normal_(m.weight, mode='fan_out')
            elif isinstance(m, nn.BatchNorm2d): nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
    def _make(self, in_c, out_c, n, stride):
        layers = [BasicBlock(in_c, out_c, stride)] + [BasicBlock(out_c, out_c, 1) for _ in range(n - 1)]
        return nn.Sequential(*layers)
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer3(self.layer2(self.layer1(x)))
        return self.fc(self.pool(x).flatten(1))
    def get_features(self, x): self(x); return self._feats
    def get_classifier_weights(self): return self.fc.weight.detach()

@torch.no_grad()
def compute_nc(model, loader, K=10):
    model.eval(); fl, ll = [], []
    for x, y in loader:
        fl.append(model.get_features(x.to(DEVICE, non_blocking=True)).cpu()); ll.append(y)
    H = torch.cat(fl).float(); Y = torch.cat(ll)
    mu_G = H.mean(0); mu_c = torch.stack([H[Y == c].mean(0) for c in range(K)])
    M = mu_c - mu_G
    Sw = sum((H[Y == c] - mu_c[c]).T @ (H[Y == c] - mu_c[c]) for c in range(K)) / len(H)
    Sb = M.T @ M / K
    nc1 = (torch.trace(Sw) / torch.trace(Sb).clamp(1e-10)).item()
    return {'nc1': nc1, 'feat_norm': H.norm(dim=1).mean().item()}

@torch.no_grad()
def eval_loss_acc(model, loader, loss_fn, K=10):
    model.eval(); tot_loss = correct = total = 0.0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        logits = model(x)
        l = (F.mse_loss(logits, F.one_hot(y, K).float(), reduction='sum')
             if loss_fn == 'mse' else F.cross_entropy(logits, y, reduction='sum'))
        tot_loss += l.item(); correct += (logits.argmax(1) == y).sum().item(); total += len(y)
    return tot_loss / total, correct / total

def weight_norm(model):
    with torch.no_grad():
        return float(torch.sqrt(sum(p.pow(2).sum() for p in model.parameters())).item())

def grad_norm_on_batch(model, x, y, loss_fn, K=10):
    model.zero_grad(set_to_none=True); model.train()
    logits = model(x.to(DEVICE, non_blocking=True))
    yb = y.to(DEVICE, non_blocking=True)
    loss = (F.mse_loss(logits, F.one_hot(yb, K).float())
            if loss_fn == 'mse' else F.cross_entropy(logits, yb))
    loss.backward()
    g = torch.sqrt(sum(p.grad.pow(2).sum() for p in model.parameters() if p.grad is not None))
    model.zero_grad(set_to_none=True)
    return float(g.item())
print('models + metrics + signal probes ready')

models + metrics + signal probes ready


In [4]:
def make_opt_sched(params, optimizer_type, n_ep, phase):
    if optimizer_type == 'adam':
        opt = torch.optim.Adam(params, lr=1e-3, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_ep)
    else:
        opt = torch.optim.SGD(params, lr=0.1, momentum=0.9, weight_decay=1e-3, nesterov=True)
        milestones = [100, 150] if phase == 1 else [300, 450]
        sch = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=milestones, gamma=0.1)
    return opt, sch

# Rich-logging two-phase runner: logs all 5 candidate predictors at each eval epoch.
def run_twophase_logged(model, train_loader, test_loader, name, optimizer_type,
                        phase1=200, phase2=600, nc_every=10, K=10):
    model = model.to(DEVICE); rows = []; terminal = False
    probe_x, probe_y = next(iter(train_loader))   # fixed batch for the gradient-norm probe
    t0 = time.time()
    for phase, loss_fn, n_ep in [(1, 'ce', phase1), (2, 'mse', phase2)]:
        opt, sch = make_opt_sched(model.parameters(), optimizer_type, n_ep, phase)
        off = phase1 if phase == 2 else 0
        for ep_l in range(1, n_ep + 1):
            ep = off + ep_l; model.train()
            for x, y in train_loader:
                x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                logits = model(x)
                loss = (F.mse_loss(logits, F.one_hot(y, K).float())
                        if loss_fn == 'mse' else F.cross_entropy(logits, y))
                if not torch.isfinite(loss):
                    print('  [%s] non-finite loss at ep %d -> aborting' % (name, ep))
                    return pd.DataFrame(rows)
                loss.backward(); opt.step()
            sch.step()
            if ep_l % nc_every == 0 or ep_l == n_ep:
                tr_loss, tr_acc = eval_loss_acc(model, train_loader, loss_fn, K)
                te_loss, te_acc = eval_loss_acc(model, test_loader,  loss_fn, K)
                if tr_acc >= 0.99 and not terminal:
                    terminal = True; print('  [%s] terminal phase at ep %d' % (name, ep))
                nc = (compute_nc(model, train_loader, K) if (terminal or phase == 2)
                      else {'nc1': None, 'feat_norm': None})
                rows.append({'epoch': ep, 'phase': phase,
                             'nc1': nc['nc1'], 'feat_norm': nc['feat_norm'],
                             'train_loss': tr_loss, 'train_acc': tr_acc,
                             'train_err': 1.0 - tr_acc, 'test_acc': te_acc,
                             'weight_norm': weight_norm(model),
                             'grad_norm': grad_norm_on_batch(model, probe_x, probe_y, loss_fn, K)})
                if nc['nc1'] is not None and nc['nc1'] < 0.01:
                    print('  [%s] NC1<0.01 at ep %d  fn=%.4f' % (name, ep, nc['feat_norm']))
    print('  [%s] done in %.1f min' % (name, (time.time() - t0) / 60))
    return pd.DataFrame(rows)
print('rich runner ready')

rich runner ready


In [5]:
# ---- run configuration ----
FAST   = True          # FAST preset: phase2=300 + 5 seeds -> one Kaggle commit comfortably <12h.
                       # Set FAST=False for the strict full-length protocol (phase2=600, as in
                       # the main paper). FAST shortens only the Phase-2 budget; MLP-5/MNIST
                       # collapses by ~epoch 340 (tNC), so phase2=300 (total 500) still reaches
                       # collapse with margin. Disclose this Phase-2 budget as a minor protocol
                       # adaptation if you report FAST runs.
PHASE2 = 300 if FAST else 600
SEEDS  = range(5)      # FAST + 5 seeds ~8-10 h on T4. range(10) is a larger study (>12 h; split it).
RUN_RESNET = False     # ResNet-20 is much slower; do NOT enable for multi-seed runs on T4.

CONFIG = [('mlp', 'adam')] + ([('resnet', 'sgd')] if RUN_RESNET else [])

def build(arch):
    if arch == 'mlp':
        tl, vl = make_loaders(mlp_tf);    return MLP(depth=5, width=512, act_cls=nn.ReLU), tl, vl
    tl, vl = make_loaders(resnet_tf);     return ResNet20(), tl, vl

runs = {}   # tag -> per-epoch DataFrame
for arch, opt_type in CONFIG:
    for seed in SEEDS:
        tag = '%s_mnist_%s_s%d' % (arch, opt_type, seed)
        print('\n=== %s  (phase2=%d) ===' % (tag, PHASE2))
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); np.random.seed(seed)
        model, tl, vl = build(arch)
        df = run_twophase_logged(model, tl, vl, tag, opt_type, phase2=PHASE2)
        df.to_csv(SAVE_DIR + 'e8_' + tag + '.csv', index=False)
        runs[tag] = df
print('\nlogged runs:', list(runs.keys()))


=== mlp_mnist_adam_s0  (phase2=300) ===


100%|██████████| 9.91M/9.91M [00:00<00:00, 32.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.05MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.88MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.31MB/s]


  [mlp_mnist_adam_s0] terminal phase at ep 10
  [mlp_mnist_adam_s0] NC1<0.01 at ep 260  fn=1.0455
  [mlp_mnist_adam_s0] NC1<0.01 at ep 270  fn=1.0472
  [mlp_mnist_adam_s0] NC1<0.01 at ep 290  fn=1.0618
  [mlp_mnist_adam_s0] NC1<0.01 at ep 300  fn=1.0661
  [mlp_mnist_adam_s0] NC1<0.01 at ep 310  fn=1.0696
  [mlp_mnist_adam_s0] NC1<0.01 at ep 320  fn=1.0713
  [mlp_mnist_adam_s0] NC1<0.01 at ep 330  fn=1.0678
  [mlp_mnist_adam_s0] NC1<0.01 at ep 340  fn=1.0975
  [mlp_mnist_adam_s0] NC1<0.01 at ep 350  fn=1.0809
  [mlp_mnist_adam_s0] NC1<0.01 at ep 360  fn=1.0896
  [mlp_mnist_adam_s0] NC1<0.01 at ep 370  fn=1.0974
  [mlp_mnist_adam_s0] NC1<0.01 at ep 380  fn=1.1011
  [mlp_mnist_adam_s0] NC1<0.01 at ep 390  fn=1.1058
  [mlp_mnist_adam_s0] NC1<0.01 at ep 400  fn=1.1037
  [mlp_mnist_adam_s0] NC1<0.01 at ep 410  fn=1.1159
  [mlp_mnist_adam_s0] NC1<0.01 at ep 420  fn=1.1077
  [mlp_mnist_adam_s0] NC1<0.01 at ep 430  fn=1.1150
  [mlp_mnist_adam_s0] NC1<0.01 at ep 440  fn=1.1185
  [mlp_mnist_adam_

In [6]:
# ---- Exp 2: multi-seed robustness (fn* mean/std/CI per config) ----
from scipy import stats as _st  # available on Colab/Kaggle
def fn_star(df, strict=0.01, relaxed=0.05):
    d = df.dropna(subset=['nc1'])
    hit = d[d.nc1 < strict]
    if len(hit): return hit.iloc[0].feat_norm, 'NC1<0.01'
    hit = d[d.nc1 < relaxed]
    if len(hit): return hit.iloc[0].feat_norm, 'NC1<0.05'
    return None, 'none'

import collections
byconf = collections.defaultdict(list)
for tag, df in runs.items():
    conf = tag.rsplit('_s', 1)[0]
    v, crit = fn_star(df)
    if v is not None: byconf[conf].append((v, crit))

print('=== fn* robustness ===')
for conf, vals in byconf.items():
    v = np.array([a for a, _ in vals]); crit = vals[0][1]
    if len(v) >= 2:
        ci = _st.t.interval(0.95, len(v)-1, loc=v.mean(), scale=_st.sem(v))
        print('%-22s fn* = %.3f +/- %.3f  (N=%d, %s, CV=%.1f%%, 95%% CI [%.3f, %.3f])'
              % (conf, v.mean(), v.std(ddof=1), len(v), crit,
                 100*v.std(ddof=1)/v.mean(), ci[0], ci[1]))
    else:
        print('%-22s fn* = %.3f  (N=1, %s)' % (conf, v.mean(), crit))

=== fn* robustness ===
mlp_mnist_adam         fn* = 1.111 +/- 0.084  (N=5, NC1<0.01, CV=7.6%, 95% CI [1.007, 1.215])


In [7]:
# ---- Exp 1: early-warning benchmark (leave-one-seed-out) ----
# For each candidate signal we calibrate a threshold on the OTHER seeds (the
# signal value at NC onset) and measure, on the held-out seed, the lead
# (T_NC - T_cross) and whether it crosses before onset. A good early-warning
# signal has a positive, CONSISTENT (low-variance) lead; a huge lead that is
# also highly variable (e.g. train_err saturating early) is NOT a good predictor.
SIGNALS = ['feat_norm', 'train_loss', 'train_err', 'weight_norm', 'grad_norm']

def t_nc(df, strict=0.01, relaxed=0.05):
    d = df.dropna(subset=['nc1'])
    for thr in (strict, relaxed):
        hit = d[d.nc1 < thr]
        if len(hit): return int(hit.iloc[0].epoch)
    return None

def cross_epoch(df, sig, level, decreasing):
    d = df.dropna(subset=[sig])
    cond = (d[sig] <= level) if decreasing else (d[sig] >= level)
    hit = d[cond]
    return int(hit.iloc[0].epoch) if len(hit) else None

import collections
groups = collections.defaultdict(dict)   # conf -> seed -> df
for tag, df in runs.items():
    conf, s = tag.rsplit('_s', 1); groups[conf][int(s)] = df

for conf, seedmap in groups.items():
    seeds = sorted(seedmap)
    print('\n=== early-warning benchmark: %s (N=%d seeds) ===' % (conf, len(seeds)))
    print('%-12s %10s %10s %12s' % ('signal', 'lead_mean', 'lead_std', 'order_acc'))
    # value of each signal at each seed's T_NC (for leave-one-out calibration)
    for sig in SIGNALS:
        at_onset, leads, correct = {}, [], 0
        for s in seeds:
            df = seedmap[s]; T = t_nc(df)
            if T is None: continue
            row = df[df.epoch == T]
            if len(row) and pd.notna(row.iloc[0][sig]): at_onset[s] = float(row.iloc[0][sig])
        if len(at_onset) < 2:
            print('%-12s %10s %10s %12s' % (sig, 'n/a', 'n/a', 'n/a')); continue
        for s in seeds:
            if s not in at_onset: continue
            df = seedmap[s]; T = t_nc(df)
            level = np.mean([at_onset[o] for o in at_onset if o != s])  # LOO threshold
            d = df.dropna(subset=[sig])
            decreasing = d[sig].iloc[-1] < d[sig].iloc[0]
            tc = cross_epoch(df, sig, level, decreasing)
            if tc is None: continue
            lead = T - tc; leads.append(lead); correct += int(lead >= 0)
        if leads:
            print('%-12s %10.1f %10.1f %11.0f%%'
                  % (sig, np.mean(leads), np.std(leads), 100*correct/len(leads)))
        else:
            print('%-12s %10s %10s %12s' % (sig, 'n/a', 'n/a', 'n/a'))
print('\nInterpretation: the best early-warning predictor has a positive lead with')
print('LOW lead_std and high order_acc. A large but high-variance lead (often')
print('train_err, which saturates at the terminal phase) is non-specific.')


=== early-warning benchmark: mlp_mnist_adam (N=5 seeds) ===
signal        lead_mean   lead_std    order_acc
feat_norm         -10.0       29.7          40%
train_loss        232.0        9.8         100%
train_err         222.0       11.7         100%
weight_norm        -2.0       13.3          60%
grad_norm         188.0        9.8         100%

Interpretation: the best early-warning predictor has a positive lead with
LOW lead_std and high order_acc. A large but high-variance lead (often
train_err, which saturates at the terminal phase) is non-specific.
